## Step4-Final-Adjustments-CMIP

Makes final adjustments to the subglacial discharge forcing files as follows:
1. Making time axis CF compliant
2. Splitting into annual files
3. Fixing attributes (which look like they were stripped out at the bias correction stage)
4. Setting final file naming convention
5. Adding a 2300 file that is the same as 2299, in the case that 2300 is missing from the CMIP output (as is the case for CESM2-WACCM) 

Creator: Donald Slater, donald.slater@ed.ac.uk

Cleaned up by Donald Slater, 10 Feb 2026.

#### Required files/inputs
- CMIP bias-corrected monthly per-basin subglacial discharge files, e.g. *Q_CESM2-WACCM_historical_SDBN1_1850_2014_biascorrected.nc*, (produced in step 3)

#### Outputs
- The final output: CMIP monthly per-basin bias-corrected subglacial discharge files, split into annual files e.g. *Q_CESM2-WACCM_historical_SDBN1_1850.nc*


### Setup

In [14]:
# define bias-corrected runoff files for CESM2-WACCM produced in step 3
Qfiles = list()
# historical
Qfiles.append('/Users/dslater2/Documents/CESM2-WACCM/historical/runoff/Q_CESM2-WACCM_historical_SDBN1_1850_2014_biascorrected.nc')
# projection (NB globus doesn't separate into projection and extension)
Qfiles.append('/Users/dslater2/Documents/CESM2-WACCM/ssp585/runoff/projection/Q_CESM2-WACCM_ssp585_SDBN1_2015_2100_biascorrected.nc')
# extension
Qfiles.append('/Users/dslater2/Documents/CESM2-WACCM/ssp585/runoff/extension/Q_CESM2-WACCM_ssp585_SDBN1_2101_2300_biascorrected.nc')

# CESM-WACCM output file for year 2299
Q2299 = '~/Documents/CESM2-WACCM/ssp585/runoff/extension/Q_CESM2-WACCM_ssp585_SDBN1_2299.nc'


### Imports

In [2]:
import xarray as xr
import cftime
import os
import numpy as np

### Define function to do the final adjustments

In [ ]:
def final_Q_adjustments(Qfile):

    # open file
    ds_Q = xr.open_dataset(Qfile,decode_times=False)

    # make timestamps cf compliant by converting from months since Jan 1850 to cftime.DatetimeNoLeap objects
    # fixes timestamp at middle of month
    months_since_jan1850 = ds_Q['time'].values
    days_in_month = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
    dates = []
    for m in months_since_jan1850:
        m = int(m)
        year  = 1850 + m // 12
        month = m % 12 + 1
        day = np.ceil(days_in_month[month-1] / 2)
        dates.append(cftime.DatetimeNoLeap(year, month, day))

    ds_Q = ds_Q.assign_coords(time=dates)

    # fix attributes that seem to have been stripped out when doing the bias correction
    ds_Q['Q'].attrs['long_name'] = 'basin subglacial discharge, monthly mean'
    ds_Q['Q'].attrs['units'] = 'm3 s-1'
    ds_Q['Q'].attrs['grid_mapping'] = 'crs'
    ds_Q['time'].attrs['standard_name'] = 'time'
    ds_Q.attrs['Conventions'] = 'CF-1.8'

    # write out files in annual chunks
    startyear = ds_Q['time'].values[0].year
    endyear = ds_Q['time'].values[-1].year
    for year in range(startyear,endyear+1):
        ds_Q_year = ds_Q.sel(time=slice(cftime.DatetimeNoLeap(year,1,1),cftime.DatetimeNoLeap(year,12,31)))
        outfile = Qfile.split('SDBN1_')[0] + 'SDBN1_' + str(year) + '.nc'
        # remove file if it exists
        if os.path.exists(outfile):
            os.remove(outfile)
        print('writing '+outfile)
        ds_Q_year.to_netcdf(outfile)

### Do the adjustments (takes a while)

In [ ]:
for Qfile in Qfiles:
    print('Processing ' + Qfile)
    final_Q_adjustments(Qfile)
    print('')

### Create a file for 2300 that is a copy of 2299 if needed (CESM2-WACCM output only goes to 2299 for some reason)

In [ ]:
# file paths
Q2299 = Qfiles[-1].split('SDBN1_')[0] + 'SDBN1_2299.nc'
Q2300 = Qfiles[-1].split('SDBN1_')[0] + 'SDBN1_2300.nc'

# if the 2300 file doesn't exist, add a 2300 file that is the same as 2299 but with updated timestamp
if not os.path.exists(Q2300):

    # datasets
    ds_2299 = xr.open_dataset(Q2299)
    ds_2300 = ds_2299.copy()

    # replace year of time coordinate to 2300
    new_times = []
    for t in ds_2299['time'].values:
        new_time = cftime.DatetimeNoLeap(2300, t.month, t.day)
        new_times.append(new_time)
    ds_2300 = ds_2300.assign_coords(time=new_times)

    # save out
    if os.path.exists(Q2300):
        os.remove(Q2300)
    ds_2300.to_netcdf(Q2300)